In [1]:
from tqdm import tqdm
import pandas as pd
from datetime import datetime, time
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os

try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm

try:
    from google.colab import userdata
    get_secret = userdata.get
except ImportError:
    # Fallback: define a dummy get_secret for local use
    def get_secret(key):
        return None

from dotenv import load_dotenv
load_dotenv()


from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

def get_env(key: str, default: str = None):
    """Tries os.getenv first, then Colab userdata if available."""
    return os.getenv(key) or get_secret(key) or default

NEO4J_URI  = get_env("NEO4J_URI")   # bolt:// or neo4j://
NEO4J_USER = get_env("NEO4J_USER")
NEO4J_PASS = get_env("NEO4J_PASS")
NEO4J_DATABASE = get_env("NEO4J_DATABASE")

DB_HOST = get_env("POSTGRES_DB_HOST")
DB_PORT = int(get_env("POSTGRES_DB_PORT"))
DB_NAME = get_env("POSTGRES_DB_NAME")
DB_USER = get_env("POSTGRES_DB_USER")
DB_PASS = get_env("POSTGRES_DB_PASS")

connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

<a href="https://colab.research.google.com/github/crezny/DATASCI266_final_project/blob/main/KG Construction/Baseline Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [2]:
import time
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM


def run_hf_llm(
    df: pd.DataFrame,
    model_path: str,                  # e.g. "meta-llama/Llama-3.2-3B-Instruct"
    prompt_col: str = "prompt_text",
    system_prompt: str = "",
    max_new_tokens: int = 256,
    temperature: float = 0.2,
    top_p: float = 0.9,
    messages: list | None = None,     # optional base messages
    chat_format: str | None = None,   # unused, kept for API compatibility
    strip_think_tags: bool = True,
) -> pd.DataFrame:
    """
    Generate text for each row in `df[prompt_col]` using a Hugging Face chat model.
    Adds columns: 'output', 'time_taken', 'token_count'.
    Works with chat-tuned models that provide a tokenizer chat_template, including:
      - HuggingFaceTB/SmolLM3-3B
      - google/gemma-2-9b-it
      - mistralai/Mistral-7B-Instruct-v0.3
      - meta-llama/Llama-3.2-3B-Instruct
    """

    # 1. Load tokenizer + model once
    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)

    torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch_dtype,
        device_map="auto",
    )
    model.eval()

    # 2. Ensure pad token is set
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id

    # Base messages (do NOT mutate user-passed list)
    base_messages = messages[:] if messages else []

    def build_prompt_text(user_prompt: str) -> str:
        """
        Build a per-row chat prompt using the model's chat template when available.
        """
        row_messages = []

        # 1) System prompt: only add if not already present in base_messages
        if system_prompt and not any(m.get("role") == "system" for m in base_messages):
            row_messages.append({"role": "system", "content": system_prompt})

        # 2) Any pre-provided messages (e.g., few-shot examples)
        row_messages.extend(base_messages)

        # 3) Row-specific user message
        row_messages.append({"role": "user", "content": user_prompt})

        # 4) Use chat template if available
        if hasattr(tokenizer, "apply_chat_template"):
            return tokenizer.apply_chat_template(
                row_messages,
                tokenize=False,
                add_generation_prompt=True,
            )

        # Fallback: plain instruct-style prompt
        parts = []
        if system_prompt:
            parts.append(system_prompt.strip())
        for m in base_messages:
            role = m.get("role", "user").upper()
            parts.append(f"{role}:\n{m.get('content', '').strip()}")
        parts.append(f"USER:\n{user_prompt.strip()}\nASSISTANT:\n")
        return "\n\n".join(parts)

    outputs = []
    time_taken = []
    token_counts = []

    # 3. Loop rows with progress bar
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Generating"):
        user_prompt = str(row[prompt_col])

        prompt_text = build_prompt_text(user_prompt)
        enc = tokenizer(prompt_text, return_tensors="pt").to(model.device)

        start = time.time()
        with torch.no_grad():
            gen_ids = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                do_sample=temperature > 0,
                pad_token_id=model.config.pad_token_id,
            )
        end = time.time()

        # Slice off prompt tokens → only generated text
        gen_only_ids = gen_ids[0][enc["input_ids"].shape[1]:]
        gen_text = tokenizer.decode(gen_only_ids, skip_special_tokens=True).strip()

        if strip_think_tags and gen_text:
            # optionally nuke <think>...</think> if a model uses that style
            # cheap + safe
            gen_text = gen_text.replace("<think>", "").replace("</think>", "").strip()

        token_count = len(tokenizer(gen_text).input_ids) if gen_text else 0

        outputs.append(gen_text)
        time_taken.append(round(end - start, 2))
        token_counts.append(token_count)

    # 4. Return df with outputs + timing info
    out_df = df.copy()
    out_df["output"] = outputs
    out_df["time_taken"] = time_taken
    out_df["token_count"] = token_counts
    return out_df


In [3]:
all_models = []

all_models.append({"model_name": "SmolLM3", "model_path": "HuggingFaceTB/SmolLM3-3B", "messages": [], "chat_format": "chatml"})
all_models.append({"model_name": "Gemma-2-9B", "model_path": "google/gemma-2-9b-it", "messages": [], "chat_format": "gemma-2"})
all_models.append({"model_name": "Mistral-7B-Instruct-v0.3", "model_path": "mistralai/Mistral-7B-Instruct-v0.3", "messages": [], "chat_format": "mistral-7b"})
all_models.append({"model_name": "Llama-3.2-3B-Instruct", "model_path": "meta-llama/Llama-3.2-3B-Instruct", "messages": [], "chat_format": "llama-3"})

In [4]:
VARIANT_TEMPLATES = {
    "A": (
        'Draft the "{section_title} (Section {section_number})" section for the "{policy_title}" '
        'appropriate for a company seeking "{framework}" compliance.\n'
        'Write in clear, enforceable language suitable for audit.\n'
        'Address, in integrated paragraph form, the following topics: purpose, acceptable use, prohibited activities, \n'
        'data handling expectations, monitoring/privileges, incident reporting, and enforcement. \n'
        'Do not label these as headings; weave them into a cohesive section. '
        'monitoring/privileges, incident reporting, and enforcement.\n'
        'Do not assume company-specific facts beyond standard "{framework}" expectations.\n'
        'Return only the section text.'
    )
    # ,
    # "B": (
    #     'Write section {section_title} (Section {section_number}) of the {policy_title} '
    #     'aligned with {framework} controls. Use concise, enforceable language and cover '
    #     'purpose, responsibilities, prohibited actions, data handling, and enforcement.'
    # ),
    # "C": (
    #     'Create section {section_number} for the {policy_title} following {framework} guidance. '
    #     'Emphasize enforceability, monitoring, and incident reporting; exclude company specifics.'
    # ),
    # "D": (
    #     'Generate subsection {section_number} ({section_title}) of the {policy_title} '
    #     'under the {framework} framework. Ensure the output is auditable and action-oriented.'
    # ),
}

In [5]:
def make_variants(row):
    vals = dict(
        section_number=row["section_number"],
        section_title=(row.get("section_title") or "").strip().capitalize(),
        policy_title=row["policy_title"],
        framework=row["source_framework"],
    )
    return [(vid, template.format(**vals).strip()) for vid, template in VARIANT_TEMPLATES.items()]

def norm_section(s: str) -> str:
    s = "" if pd.isna(s) else str(s).strip()
    s = re.sub(r"[^0-9.]+", "", s)
    s = re.sub(r"\.+", ".", s).strip(".")
    return s

In [6]:
def get_section_text(df, policy_id: str, section: str, include_children=True) -> str:
    # Filter for this policy
    pdf = df[df['policy_id'] == policy_id].copy()
    if pdf.empty:
        return f"[No content found for policy {policy_id}]"

    # Choose filtering style: exact match or include children (prefix)
    if include_children:
        section_df = pdf[pdf['clause_section_number'].str.startswith(section)]
    else:
        section_df = pdf[pdf['clause_section_number'] == section]

    if section_df.empty:
        return f"[No section {section} found for policy {policy_id}]"

    # # Sort numerically by section number (handles 5, 5.1, 5.2, 5.10 correctly)
    # section_df = section_df.sort_values(
    #     by="clause_section_number",
    #     key=lambda s: s.str.split('.').apply(lambda x: [int(i) for i in x])
    # )

    output_lines = []
    last_section = None

    for _, row in section_df.iterrows():
        sec_num = row['clause_section_number']
        title = row.get('section_title', '')
        text = str(row.get('clause_text', '')).strip()

        # Print header at first appearance of each section/subsection
        if sec_num != last_section:
            header = f"{sec_num} {title}".strip()
            output_lines.append(header)
            last_section = sec_num

        # Add the body text
        if text:
            output_lines.append(text)

    return "\n".join(output_lines)


In [7]:

engine = create_engine(connection_url, echo=False)
_org_df = pd.read_sql("SELECT * FROM policy_lines where policy_origin = 'client'", engine)

org_df = _org_df[['policy_id','policy_title','source_framework','policy_origin','section_id','section_title','clause_section_number','company_type','company_id']]
org_df = org_df.drop_duplicates()
org_df['section_text'] = org_df[['policy_id', 'clause_section_number']].apply(lambda row: get_section_text(_org_df, row['policy_id'], row['clause_section_number']),axis =1)
# org_df = org_df.sample(n=1, random_state=42)

org_df["section_number"] = org_df["clause_section_number"].apply(norm_section)

records = []
for _, r in org_df.iterrows():
    for vid, prompt in make_variants(r):
        records.append({
            "policy_id": r["policy_id"],
            "policy_title": r["policy_title"],
            "section_number": r["section_number"],
            "variant": vid,
            "prompt_text": prompt
        })

prompts_variants_df = pd.DataFrame(records)


In [8]:
sp = (
    "You are a policy-writing assistant tasked with drafting clear, enforceable cybersecurity and IT governance policies. "
    "Each response should align with the specified frameworks expectations. "
    "Use concise, professional language appropriate for inclusion in a formal company policy. "
    "Do not include any explanations or justifications for the policy content. "
    "Do not include the section title or section number like (3.1, 2.2, ETC.) in your response. "
    "Do not include prefaces or meta text — return only the policy section text itself."
)
df = prompts_variants_df.copy()
# df = df.sample(n=1, random_state=42)

# Ensure consistent types
df["policy_id"] = df["policy_id"].astype(str)

# Unique prompt_text -> all policy_ids that use it
prompt_map = (
    df.groupby("prompt_text", as_index=False)
      .agg(policy_ids=("policy_id", lambda s: sorted(s.unique())))
)

# Sample the subset of prompts you actually want to score
# (shared across all models, so you can compare apples-to-apples)
# sampled_prompt_map = prompt_map.sample(n=200, random_state=42)
sampled_prompt_map = prompt_map.copy()

# LLM inputs: one row per sampled prompt_text
llm_inputs = sampled_prompt_map[["prompt_text"]].copy()

all_results = []  # collect per-model mappings here

for m in all_models:
    model_name  = m["model_name"]
    model_path  = m["model_path"]
    messages    = m.get("messages", None)
    chat_format = m.get("chat_format", None)
    if model_name == "Gemma-2-9B":
      system_prompt = None
    else:
      system_prompt = sp

    print(f"Scoring {len(llm_inputs)} prompts with {model_name} ...")

    # Run your baseline / HF model on the sampled unique prompts
    out_df = run_hf_llm(
        df=llm_inputs.copy(),
        model_path=model_path,
        max_new_tokens=256,
        temperature=0.2,
        top_p=0.9,
        messages=messages,              # <-- model-specific system/user messages
        chat_format=chat_format,   # <-- model-specific chat_format
        prompt_col="prompt_text",
        system_prompt=system_prompt,
        strip_think_tags=True,
    )

    # out_df must include: 'prompt_text', 'output', 'time_taken', 'token_count'
    if "prompt_text" not in out_df.columns:
        raise ValueError(f"{model_name}: out_df missing 'prompt_text' column")

    # Attach the policy_ids list for each prompt, then explode
    model_mapping = (
        out_df.merge(sampled_prompt_map, on="prompt_text", how="left")
              .explode("policy_ids")
              .rename(columns={"policy_ids": "policy_id"})
    )

    # Normalize types & tag with model
    model_mapping["policy_id"]   = model_mapping["policy_id"].astype(str)
    model_mapping["model_name"]  = model_name

    # Keep just what we need for the final join
    model_mapping = model_mapping[[
        "prompt_text",
        "policy_id",
        "model_name",
        "output",
        "time_taken",
        "token_count",
    ]]

    all_results.append(model_mapping)

# Stack all models' mappings together
all_mappings = pd.concat(all_results, ignore_index=True)



Scoring 1144 prompts with SmolLM3 ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Generating:   0%|          | 0/1144 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
base = (
    df.assign(key=1)
      .merge(
          pd.DataFrame({"model_name": [m["model_name"] for m in all_models], "key": 1}),
          on="key",
          how="left"
      )
      .drop(columns="key")
)

final = (
    all_mappings.merge(
            base,
            on=["prompt_text", "policy_id", "model_name"],
            how="left"  # rows for non-sampled prompts stay NaN/None
        )
        .sort_values(["model_name", "policy_id", "section_number", "variant"], kind="stable")
        .reset_index(drop=True)
)
final['model_type'] = 'baseline'
final.drop(columns=['prompt_text'], inplace=True)
final.to_sql('model_results2', engine, if_exists='replace', index=False)

In [ ]:
output_mg_test = pd.read_sql('select * from model_results2 ', engine)
output_mg_test
len(output_mg_test)

In [ ]:
output_mg_test